# 01b · Finance billing: bronze and silver
Loads the finance billing feed written by `00_simulate_finance_billing`.
Like the platform sources, each run lands a full snapshot in a new `ingest_date=` folder, and only the **latest** one is loaded,
so daily runs never duplicate invoices. Duplicate invoices *inside* a snapshot are kept and flagged, not deleted.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.bronze_finance_billing AS
WITH files AS (
  SELECT *,
         _metadata.file_path                                        AS _source_file,
         regexp_extract(_metadata.file_path, 'ingest_date=([0-9-]+)', 1) AS _ingest_date
  FROM read_files(
    '/Volumes/workspace/media_pipeline/bronze/finance_billing/*/*.csv',
    format => 'csv', header => true, inferColumnTypes => false
  )
)
SELECT * EXCEPT (_ingest_date), current_timestamp() AS _ingested_at
FROM files
WHERE _ingest_date = (SELECT MAX(_ingest_date) FROM files);

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.media_pipeline.silver_finance_billing AS
SELECT
  invoice_id,
  vendor                                        AS platform,
  billing_month,
  CAST(amount AS DECIMAL(12,2))                 AS amount,
  currency,
  COUNT(*) OVER (PARTITION BY invoice_id) > 1   AS is_duplicate_invoice,
  _source_file
FROM workspace.media_pipeline.bronze_finance_billing;

In [0]:
%sql
-- Expected: 15 invoice lines, 2 duplicate lines (one invoice sent twice), every run
SELECT COUNT(*)                       AS invoice_lines,
       COUNT_IF(is_duplicate_invoice) AS duplicate_lines,
       SUM(amount)                    AS total_billed
FROM workspace.media_pipeline.silver_finance_billing;